# Week 4, days 3–5 on a Mac: porting Python to C++ with open-source models

The course labs (`day3.ipynb`, `day4.ipynb`, `day5.ipynb`) use GPT-5, Claude, Gemini and Grok, which need API keys.
Here the same experiments run on the models pulled into Ollama, using the shared helpers in
[code_porter.py](code_porter.py).

Two things differ from the course, and both came out of trying it locally:

1. **Verify and fix.** The course writes the model's reply to `main.cpp` and hopes it compiles. Local models
   rarely get it right first time, so `port_and_verify` compiles the code, runs it, compares its output with the
   Python's, and sends any compiler error or wrong answer back to the model for another try.
2. **No `-Ofast`.** Fast-math lets the compiler reorder floating-point additions. That changed the 12th decimal
   of the pi result, so a correct port looked wrong. Plain `-O3` keeps the arithmetic identical to Python's.

Start Ollama first (`ollama serve`). `qwen2.5:14b` is the workhorse here; if you `ollama pull qwen2.5-coder:14b`
or `gpt-oss:20b`, they appear in the leaderboards automatically.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from code_porter import (
    PI, PYTHON_HARD, available_languages, available_models, compile_and_run, compile_command,
    port, port_until_verified, run_python, source_file, speedup_note, timing,
)
from system_info import retrieve_system_info, rust_toolchain_info

models = list(available_models())
system_info = retrieve_system_info()
print("Models:", models)
print("Languages I can compile:", available_languages())

Models: ['Ollama: llama3.2:latest', 'Ollama: qwen2.5:14b']
Languages I can compile: ['C++']


## Day 3 – the compile command

The course asks GPT-5 which compiler flags to use. On an Apple Silicon Mac with the Xcode command line tools,
`clang++` is already there; `compile_command` builds the command for whatever machine it runs on.

In [2]:
print(system_info["cpu"])
print(" ".join(compile_command("C++")))

{'brand': 'Apple M4 Pro', 'cores_logical': 12, 'cores_physical': 12, 'simd': []}
/usr/bin/clang++ -std=c++17 -O3 -mcpu=native -flto=thin -DNDEBUG /Users/eugene_dyl/PycharmProjects/llm_engineering/week4/build/main.cpp -o /Users/eugene_dyl/PycharmProjects/llm_engineering/week4/build/main


## Day 3 – the pi series in Python

200 million iterations of a Leibniz-style series. This is the baseline every port is measured against.

In [3]:
pi_expected = run_python(PI)
print(pi_expected)

Result: 3.141592656089
Execution Time: 10.513788 seconds



## Day 3 – port it the course's way: one shot, no checking

`port` does exactly what the course's `port` does: one request, strip the code fences, done.

In [4]:
LOCAL_MODEL = "Ollama: qwen2.5:14b"

cpp = port(LOCAL_MODEL, PI, "C++", system_info)
print(cpp)
print(compile_and_run(cpp, "C++"))

#include <iostream>
#include <chrono>

double calculate(long long iterations, long long param1, long long param2) {
    double result = 1.0;
    for (long long i = 1; i <= iterations; ++i) {
        long long j = i * param1 - param2;
        result -= (1.0 / j);
        j = i * param1 + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    auto start_time = std::chrono::high_resolution_clock::now();
    double result = calculate(200000000, 4, 1) * 4;
    auto end_time = std::chrono::high_resolution_clock::now();

    auto duration = std::chrono::duration_cast<std::chrono::duration<double>>(end_time - start_time);

    std::cout << "Result: " << std::fixed << std::setprecision(12) << result << std::endl;
    std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << duration.count() << " seconds" << std::endl;

    return 0;
}



An error occurred while compiling:
main.cpp:22:51: error: no member named 'setprecision' in namespace 'std'
   22 |     std::cout << "Result: " << std::fixed << std::setprecision(12) << result << std::endl;
      |                                                   ^~~~~~~~~~~~
main.cpp:23:59: error: no member named 'setprecision' in namespace 'std'
   23 |     std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << duration.count() << " seconds" << std::endl;
      |                                                           ^~~~~~~~~~~~
2 errors generated.



That may or may not compile – when this notebook was first run, qwen2.5:14b forgot `#include <iomanip>`.
## Day 3 – port it with verify-and-fix

Now the compiler's complaint goes back to the model, and so does the output if it doesn't match the Python's.

In [5]:
ok, cpp, output, attempts = port_until_verified(LOCAL_MODEL, PI, "C++", system_info, expected=pi_expected)
print()
print(output, speedup_note(pi_expected, output))

Attempt 1: asking Ollama: qwen2.5:14b for C++...


  ✗ does not compile:
main.cpp:22:51: error: no member named 'setprecision' in namespace 'std'
main.cpp:23:59: error: no member named 'setprecision' in namespace 'std'
2 errors generated.


Attempt 2: asking Ollama: qwen2.5:14b for C++...


  ✓ compiles, runs and matches the Python output



Result: 3.141592656089
Execution Time: 0.243030 seconds
 
→ 43x faster than the Python


## Day 4 – a leaderboard of the open-source models

The course compares Qwen 2.5 Coder, DeepSeek Coder v2, gpt-oss and Qwen3 Coder. This runs every model you have
in Ollama through the same verify-and-fix loop and records how many attempts it needed and how much faster the
result is than Python.

In [6]:
verified_code = {}  # (model, language, first line of the Python) -> the port that passed

def leaderboard(python, expected, language="C++", model_labels=None):
    rows = []
    for label in model_labels or [m for m in models if m.startswith("Ollama")]:
        print(f"\n=== {label}")
        ok, code, output, attempts = port_until_verified(label, python, language, system_info, expected=expected)
        if ok:
            verified_code[(label, language, python.splitlines()[0])] = code
        seconds = timing(output) if ok else None
        rows.append({
            "model": label.removeprefix("Ollama: "),
            "verified": ok,
            "attempts": attempts,
            "seconds": seconds,
            "speedup": round(timing(expected) / seconds) if seconds else None,
        })
    return pd.DataFrame(rows).sort_values("speedup", ascending=False, na_position="last")

pi_board = leaderboard(PI, pi_expected)


=== Ollama: llama3.2:latest
Attempt 1: asking Ollama: llama3.2:latest for C++...


  ✗ does not compile:
main.cpp:24:34: error: no matching literal operator for call to 'operator""_000_000' with argument of type 'unsigned long long' or 'const char *', and no matching literal operator template
main.cpp:33:36: error: use of undeclared identifier 'setprecision'
main.cpp:34:44: error: use of undeclared identifier 'setprecision'
3 errors generated.


Attempt 2: asking Ollama: llama3.2:latest for C++...


  ✗ does not compile:
main.cpp:25:34: error: no matching literal operator for call to 'operator""_000_000' with argument of type 'unsigned long long' or 'const char *', and no matching literal operator template
1 error generated.


Attempt 3: asking Ollama: llama3.2:latest for C++...


  ✗ runs but prints something different:
Result: 12.566370624356
Execution Time: 0.244593 seconds


... (3 outputs left out)


=== Ollama: qwen2.5:14b
Attempt 1: asking Ollama: qwen2.5:14b for C++...


  ✗ does not compile:
main.cpp:22:51: error: no member named 'setprecision' in namespace 'std'
main.cpp:23:59: error: no member named 'setprecision' in namespace 'std'
2 errors generated.


Attempt 2: asking Ollama: qwen2.5:14b for C++...


  ✓ compiles, runs and matches the Python output


In [7]:
pi_board

,model,verified,attempts,seconds,speedup
1,qwen2.5:14b,True,2,0.236362,44.0
0,llama3.2:latest,False,4,NaN,NaN


## Day 4/5 – the Gradio app

The same UI as day 5 (the course's `styles.py`), with a model dropdown of whatever is available, a C++/Rust
switch, and a **Verify & fix** checkbox. The porting log shows each attempt as it happens, and **Run port**
reports the speedup over the Python run. You can also run it on its own: `python week4/code_porter.py`.

In [ ]:
from code_porter import build_ui

ui = build_ui()
ui.launch(inbrowser=True)

## Day 5 – Rust

The course switches to Rust on day 5. `rust_toolchain_info` reports whether a toolchain is installed; if it
isn't, install it from https://rustup.rs and the Rust option appears in `available_languages()` and in the app.

In [9]:
rust = rust_toolchain_info()
print("Rust installed:", rust["installed"])
LANGUAGE = "Rust" if "Rust" in available_languages() else "C++"
print("Porting the day 5 example to", LANGUAGE)

Rust installed: False
Porting the day 5 example to C++


## Day 5 – the hard one: maximum subarray sums

An LCG random generator plus an O(n²) search, 20 times over. The traps are the ones the course warns about:
Python integers never overflow, and the LCG has to produce exactly the same sequence.

In [10]:
hard_expected = run_python(PYTHON_HARD)
print(hard_expected)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 19.898077 seconds



In [11]:
hard_board = leaderboard(PYTHON_HARD, hard_expected, LANGUAGE)
hard_board


=== Ollama: llama3.2:latest
Attempt 1: asking Ollama: llama3.2:latest for C++...


  ✗ runs but prints something different:
Total Maximum Subarray Sum (20 runs): 14584
Execution Time: 274 milliseconds


Attempt 2: asking Ollama: llama3.2:latest for C++...


  ✗ runs but prints something different:
Total Maximum Subarray Sum (20 runs): 12119
Execution Time: 265 milliseconds


Attempt 3: asking Ollama: llama3.2:latest for C++...


  ✗ does not compile:
main.cpp:54:20: error: use of undeclared identifier 'randomNumbers'
1 warning and 1 error generated.


... (2 outputs left out)

Gave up after 4 attempts.



=== Ollama: qwen2.5:14b
Attempt 1: asking Ollama: qwen2.5:14b for C++...


  ✓ compiles, runs and matches the Python output


,model,verified,attempts,seconds,speedup
1,qwen2.5:14b,True,1,0.525052,38.0
0,llama3.2:latest,False,4,NaN,NaN


## What the verified port looks like

In [12]:
best = hard_board.iloc[0]
if best["verified"]:
    code = verified_code[(f"Ollama: {best['model']}", LANGUAGE, PYTHON_HARD.splitlines()[0])]
    display(Markdown(f"**{best['model']}**, {best['speedup']:,.0f}x faster than Python:"))
    display(Markdown(f"```{'rust' if LANGUAGE == 'Rust' else 'cpp'}\n{code}\n```"))

**qwen2.5:14b**, 38x faster than Python:

```cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdint>

uint64_t lcg(uint64_t seed, uint64_t a = 1664525, uint64_t c = 1013904223, uint64_t m = 4294967296) {
    return (a * seed + c) % m;
}

std::vector<int64_t> generate_random_numbers(uint64_t seed, int64_t min_val, int64_t max_val, size_t n) {
    std::vector<int64_t> random_numbers;
    uint64_t value = seed;
    for (size_t i = 0; i < n; ++i) {
        value = lcg(value);
        random_numbers.push_back((value % (max_val - min_val + 1)) + min_val);
    }
    return random_numbers;
}

int64_t max_subarray_sum(const std::vector<int64_t>& random_numbers) {
    int64_t max_sum = INT64_MIN;
    for (size_t i = 0; i < random_numbers.size(); ++i) {
        int64_t current_sum = 0;
        for (size_t j = i; j < random_numbers.size(); ++j) {
            current_sum += random_numbers[j];
            if (current_sum > max_sum) {
                max_sum = current_sum;
            }
        }
    }
    return max_sum;
}

int64_t total_max_subarray_sum(uint64_t initial_seed, int64_t min_val, int64_t max_val, size_t n, size_t runs = 20) {
    int64_t total_sum = 0;
    uint64_t seed = initial_seed;
    for (size_t i = 0; i < runs; ++i) {
        seed = lcg(seed);
        total_sum += max_subarray_sum(generate_random_numbers(seed, min_val, max_val, n));
    }
    return total_sum;
}

int main() {
    const size_t n = 10000;         // Number of random numbers
    const uint64_t initial_seed = 42; // Initial seed for the LCG
    const int64_t min_val = -10;    // Minimum value of random numbers
    const int64_t max_val = 10;     // Maximum value of random numbers

    auto start_time = std::chrono::high_resolution_clock::now();
    int64_t result = total_max_subarray_sum(initial_seed, min_val, max_val, n);
    auto end_time = std::chrono::high_resolution_clock::now();

    auto duration = std::chrono::duration_cast<std::chrono::duration<double>>(end_time - start_time);

    std::cout << "Total Maximum Subarray Sum (20 runs): " << result << std::endl;
    std::cout << "Execution Time: " << duration.count() << " seconds" << std::endl;

    return 0;
}

```

## Results on an M4 Pro (Ollama, C++ with clang -O3)

| Task | Python | qwen2.5:14b | llama3.2 (3B) |
|---|---|---|---|
| Pi series (day 3/4) | 10.5 s | **0.24 s, 44x** – needed 2 attempts | failed all 4 attempts |
| Max subarray sum (day 5) | 19.9 s | **0.53 s, 38x** – first attempt | failed all 4 attempts |

For comparison, in Ed's runs the frontier models reached 184–1440x on pi; the gap is mostly that they pick
cleverer algorithms (Kadane's O(n) scan for the subarray problem), while qwen keeps the O(n²) loops and only wins
by compiling them.

What the verify-and-fix loop taught:

- **One shot is not enough locally.** qwen2.5:14b forgot `#include <iomanip>` for pi in every run; llama3.2 wrote
  Python's `200_000_000` literal into C++. Both are trivial once the compiler error goes back to the model.
- **Vague feedback makes a small model worse.** Told only "the output is wrong", qwen printed `3.14159` and then
  "fixed" it by breaking the formula. `mismatch_hints` now spells out the exact line, the number of decimals, and
  that `18446744073709551596` is −20 in an unsigned 64-bit type – after that, 6 of 6 repeat runs passed.
- **Temperature matters for a leaderboard.** At Ollama's default temperature one full run of this notebook had
  qwen failing both tasks; at 0.2 the results above repeated every time.
- **Context length.** Ollama's OpenAI-compatible endpoint is capped at 4096 tokens here and silently drops the start
  of long conversations, so [llm.py](llm.py) calls the native API with a 16k context.
- **3B is too small for this job.** llama3.2 never verified a port: wrong algorithms, undeclared variables, and the
  same error repeated across attempts.
